# RAG Pipeline — Multi-Source (PDF + URL + CSV)
Supports structured and unstructured CSV files alongside PDF and URL sources.

## IMPORTS

In [10]:
import os
import re
import pandas as pd
from dotenv import load_dotenv
from collections import defaultdict

from langchain_community.document_loaders import PyPDFLoader, CSVLoader, WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from llama_index.core import Document
from llama_index.core.node_parser import SentenceWindowNodeParser
from llama_index.core import VectorStoreIndex, KnowledgeGraphIndex
from llama_index.core import Document as LlamaDocument
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.groq import Groq

from sentence_transformers import CrossEncoder
from langchain_groq import ChatGroq

## CSV LOADER UTILITY

Two strategies depending on CSV type:
- **Text/Unstructured CSV** → use `CSVLoader` (columns contain sentences/paragraphs)
- **Numeric/Tabular CSV** → use `PandasCSVLoader` (converts each row into a readable text summary)

In [11]:
def load_csv_smart(csv_path, text_columns=None, label_column=None, max_rows=None):
    """
    Smart CSV loader — handles both text and numeric CSV files.

    Parameters:
    -----------
    csv_path     : str  — path to your CSV file
    text_columns : list — columns that contain meaningful text (for unstructured CSVs)
                          If None, all columns are used.
    label_column : str  — a column to use as a label/title for each row (optional)
    max_rows     : int  — limit rows to avoid huge docs (optional, e.g. 500)

    Returns:
    --------
    List of LangChain Document objects
    """
    df = pd.read_csv(csv_path)

    if max_rows:
        df = df.head(max_rows)

    # If user specified text columns, use only those
    if text_columns:
        cols_to_use = [c for c in text_columns if c in df.columns]
    else:
        cols_to_use = list(df.columns)

    docs = []

    for i, row in df.iterrows():
        # Build readable text from selected columns
        parts = []
        for col in cols_to_use:
            val = row[col]
            if pd.notna(val) and str(val).strip() != "":
                parts.append(f"{col}: {val}")

        page_content = "\n".join(parts)

        if not page_content.strip():
            continue  # skip empty rows

        metadata = {
            "source": csv_path,
            "source_type": "csv",
            "row_index": i,
        }

        # Optionally tag with a label column
        if label_column and label_column in df.columns:
            metadata["label"] = str(row[label_column])

        docs.append(Document(page_content=page_content, metadata=metadata))

    print(f"[CSV] Loaded {len(docs)} documents from '{csv_path}'")
    return docs

## MULTIPLE SOURCES LOADING

In [12]:
url = "https://www.geeksforgeeks.org/artificial-intelligence/artificial-intelligence"

url_loader = WebBaseLoader(url)
pdf_loader = PyPDFLoader("C:\\Users\\yoush\\OneDrive\\Desktop\\3GenAIProj\\data\\SDG.pdf")

url_docs = url_loader.load()
pdf_docs = pdf_loader.load()

In [13]:
# ─────────────────────────────────────────────────────────────────
# CSV LOADING — choose the right strategy for your file:
# ─────────────────────────────────────────────────────────────────

# ── OPTION A: Text/Unstructured CSV (e.g. news articles, reviews)
# Use this when columns contain actual readable sentences.
#
# Example CSV structure:
#   title         | content                        | category
#   AI in 2025    | Artificial intelligence has... | tech
# #
# csv_docs = load_csv_smart(
#     csv_path="data/articles.csv",
#     text_columns=["title", "content"],   # columns with real text
#     label_column="category",             # optional: tag each doc
# )


# ── OPTION B: Numeric/Tabular CSV (e.g. MNIST, sales, sensor data)
# Use this when columns contain numbers, IDs, or short codes.
# Each row becomes a short textual summary for embedding.
#
# Example CSV structure:
#   label | pixel1 | pixel2 | ...
#   5     | 0      | 128    | ...
#
# csv_docs = load_csv_smart(
#     csv_path="data/mnist_train.csv",
#     text_columns=["label"],   # only meaningful column for RAG
#     label_column="label",
#     max_rows=500              # limit rows for large files
# )


# ── OPTION C: General CSV (use all columns, no filtering)
# Best when you're unsure — uses all columns as key:value text.
#
csv_docs = load_csv_smart(
    csv_path="C:\\Users\\yoush\\OneDrive\\Desktop\\3GenAIProj\\data\\mnist_train.csv",
    text_columns=None,    # None = use all columns
    label_column=None,
    max_rows=300          # safety limit for large CSVs
)

[CSV] Loaded 300 documents from 'C:\Users\yoush\OneDrive\Desktop\3GenAIProj\data\mnist_train.csv'


## UNIFIED DOCUMENT DB

In [14]:
# Metadata Tagging
for doc in url_docs:
    doc.metadata["source_type"] = "url"

for doc in pdf_docs:
    doc.metadata["source_type"] = "pdf"

# csv_docs already tagged with source_type = "csv" inside load_csv_smart()

# Unified Document DB
all_docs = url_docs + pdf_docs + csv_docs

print(f"Total documents: {len(all_docs)}")
print(f"  URL  : {len(url_docs)}")
print(f"  PDF  : {len(pdf_docs)}")
print(f"  CSV  : {len(csv_docs)}")

Total documents: 325
  URL  : 1
  PDF  : 24
  CSV  : 300


## CHUNKING

In [ ]:
# Cleaning
for doc in all_docs:
    text = doc.page_content
    text = re.sub(r"\s+", " ", text)
    doc.page_content = text.strip()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunked_docs = splitter.split_documents(all_docs)
print(f"Total chunks after splitting: {len(chunked_docs)}")

TypeError: expected string or bytes-like object, got 'Document'

## EMBEDDING

In [ ]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

## GROQ API LOADING

In [ ]:
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

Settings.llm = Groq(
    model="llama-3.3-70b-versatile",
    api_key=api_key
)

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=api_key
)

## INDEXING LAYER
### FAISS Vector DB

In [ ]:
db = FAISS.from_documents(chunked_docs, embedding)

retriever1 = db.as_retriever(similarity_top_k=5)

### SENTENCE WINDOW INDEX

In [ ]:
llama_docs = [
    LlamaDocument(text=doc.page_content, metadata=doc.metadata)
    for doc in chunked_docs
]

node_parser = SentenceWindowNodeParser.from_defaults(
    window_size=3,
    window_metadata_key="window",
    original_text_metadata_key="original_text"
)

nodes = node_parser.get_nodes_from_documents(llama_docs)
sentence_index = VectorStoreIndex(nodes)
retriever2 = sentence_index.as_retriever(similarity_top_k=5)

### KNOWLEDGE GRAPH INDEX

In [ ]:
graph_index = KnowledgeGraphIndex.from_documents(
    llama_docs,
    max_triplets_per_chunk=10
)
retriever3 = graph_index.as_retriever(include_text=True)

## RETRIEVAL

In [ ]:
query = "What is Artificial Intelligence?"

vector_results   = retriever1.invoke(query)
sentence_results = retriever2.retrieve(query)
graph_results    = retriever3.retrieve(query)

In [ ]:
retrieved_docs = []

# Vector retriever results
for rank, doc in enumerate(vector_results):
    retrieved_docs.append({
        "text": doc.page_content,
        "metadata": doc.metadata,
        "rank": rank,
        "source": "vector"
    })

# Sentence window retriever results
for rank, node in enumerate(sentence_results):
    retrieved_docs.append({
        "text": node.node.text,
        "metadata": node.node.metadata,
        "rank": rank,
        "source": "sentence_window"
    })

# Knowledge graph retriever results
for rank, node in enumerate(graph_results):
    retrieved_docs.append({
        "text": node.get_content(),
        "metadata": node.metadata,
        "rank": rank,
        "source": "knowledge_graph"
    })

## RECIPROCAL RANK FUSION (RRF)

In [ ]:
k = 60

rrf_scores = defaultdict(lambda: {"score": 0, "metadata": None, "source": None})

for doc in retrieved_docs:
    text = doc["text"]
    rrf_scores[text]["score"]    += 1 / (k + doc["rank"])
    rrf_scores[text]["metadata"]  = doc["metadata"]
    rrf_scores[text]["source"]    = doc["source"]

fused_results = sorted(
    rrf_scores.items(),
    key=lambda x: x[1]["score"],
    reverse=True
)

TOP_FUSION = 20
candidate_docs = fused_results[:TOP_FUSION]

## RERANKING

In [ ]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

pairs  = [(query, doc[0]) for doc in candidate_docs]
scores = reranker.predict(pairs)

reranked_docs = []
for (text, info), score in zip(candidate_docs, scores):
    reranked_docs.append({
        "text":          text,
        "rerank_score":  float(score),
        "fusion_score":  info["score"],
        "source":        info["source"],
        "metadata":      info["metadata"],
    })

reranked_docs = sorted(reranked_docs, key=lambda x: x["rerank_score"], reverse=True)

TOP_CONTEXT = 5
top_docs = reranked_docs[:TOP_CONTEXT]

## PROMPT + LLM CALL

In [ ]:
context = "\n\n".join(doc["text"] for doc in top_docs)

prompt = f"""
You are an expert AI assistant. Use ONLY the provided context to answer the question.
Response rules:
- If the question is simple: answer in 1-2 lines with a daily life example.
- If the question is detailed: use sections (DEFINITION, DESCRIPTION, EXAMPLE) with bullet points.
- If the answer is not in the context, reply: "I couldn't find the answer in the provided documents."

==========================
Context
==========================
{context}

==========================
Question
==========================
{query}

==========================
Answer
==========================
"""

response = Settings.llm.complete(prompt)
print(response.text)

## TOP RETRIEVED DOCUMENTS (Debug View)

In [ ]:
for i, doc in enumerate(top_docs, start=1):
    print(f"Rank         : {i}")
    print(f"Retriever    : {doc['source']}")
    print(f"Fusion Score : {doc['fusion_score']:.6f}")
    print(f"Rerank Score : {doc['rerank_score']:.4f}")
    print(f"Source Type  : {doc['metadata'].get('source_type', 'unknown')}")
    print(f"Metadata     : {doc['metadata']}")
    print("-" * 80)